In [1]:
import random
from datetime import date, timedelta

import torch
import torch.nn as nn
import torch.optim as optim


# Make our random results more reproducible
random.seed(42)
torch.manual_seed(42)


# Use GPU if available, otherwise CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [2]:
# Create the complete dataset

start_date = date(2020, 1, 1)
end_date = date(2030, 12, 31)

dataset = []

current_date = start_date

while current_date <= end_date:

    input_text = (
        f"{current_date.day} "
        f"{current_date.strftime('%B').lower()} "
        f"{current_date.year}"
    )

    target_text = current_date.strftime("%Y-%m-%d")

    dataset.append((input_text, target_text))

    current_date += timedelta(days=1)


print("Total examples:", len(dataset))

print("\nFirst 5 examples:")

for example in dataset[:5]:
    print(example)

Total examples: 4018

First 5 examples:
('1 january 2020', '2020-01-01')
('2 january 2020', '2020-01-02')
('3 january 2020', '2020-01-03')
('4 january 2020', '2020-01-04')
('5 january 2020', '2020-01-05')


In [3]:
# Shuffle and split the dataset

random.shuffle(dataset)

train_size = int(0.80 * len(dataset))
validation_size = int(0.10 * len(dataset))

train_data = dataset[:train_size]

validation_data = dataset[
    train_size:
    train_size + validation_size
]

test_data = dataset[
    train_size + validation_size:
]


print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))
print("Test examples:", len(test_data))

Training examples: 3214
Validation examples: 401
Test examples: 403


In [4]:
# Create encoder vocabulary
input_characters = list(
    "0123456789 abcdefghijklmnopqrstuvwxyz"
)

input_char_to_id = {
    character: index
    for index, character in enumerate(input_characters)
}

print("Input vocabulary size:", len(input_characters))
print(input_char_to_id)

Input vocabulary size: 37
{'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, ' ': 10, 'a': 11, 'b': 12, 'c': 13, 'd': 14, 'e': 15, 'f': 16, 'g': 17, 'h': 18, 'i': 19, 'j': 20, 'k': 21, 'l': 22, 'm': 23, 'n': 24, 'o': 25, 'p': 26, 'q': 27, 'r': 28, 's': 29, 't': 30, 'u': 31, 'v': 32, 'w': 33, 'x': 34, 'y': 35, 'z': 36}


In [5]:
# Create decoder vocabulary

output_tokens = [
    "<START>",
    "<END>",
    "0", "1", "2", "3", "4",
    "5", "6", "7", "8", "9",
    "-"
]

output_token_to_id = {
    token: index
    for index, token in enumerate(output_tokens)
}

output_id_to_token = {
    index: token
    for token, index in output_token_to_id.items()
}

print("Output vocabulary size:", len(output_tokens))
print(output_token_to_id)

Output vocabulary size: 13
{'<START>': 0, '<END>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12}


In [6]:
def encode_input(input_text):

    ids = [
        input_char_to_id[character]
        for character in input_text
    ]

    return torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    )


def prepare_decoder_sequences(target_text):

    # What decoder receives during training
    decoder_input_tokens = (
        ["<START>"] + list(target_text)
    )

    # What decoder should predict
    decoder_target_tokens = (
        list(target_text) + ["<END>"]
    )

    decoder_input_ids = [
        output_token_to_id[token]
        for token in decoder_input_tokens
    ]

    decoder_target_ids = [
        output_token_to_id[token]
        for token in decoder_target_tokens
    ]

    decoder_input_tensor = torch.tensor(
        decoder_input_ids,
        dtype=torch.long,
        device=device
    )

    decoder_target_tensor = torch.tensor(
        decoder_target_ids,
        dtype=torch.long,
        device=device
    )

    return decoder_input_tensor, decoder_target_tensor

In [7]:
x = encode_input("29 december 2029")

decoder_x, decoder_y = prepare_decoder_sequences(
    "2029-12-29"
)

print("")
print("Encoder input IDs", x)
print("Decoder input IDs : " ,decoder_x)
print("Decoder target IDs : ",decoder_y)


Encoder input IDs tensor([ 2,  9, 10, 14, 15, 13, 15, 23, 12, 15, 28, 10,  2,  0,  2,  9])
Decoder input IDs :  tensor([ 0,  4,  2,  4, 11, 12,  3,  4, 12,  4, 11])
Decoder target IDs :  tensor([ 4,  2,  4, 11, 12,  3,  4, 12,  4, 11,  1])


In [8]:
class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_size,
        hidden_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size
        )


    def forward(self, input_tensor):

        embedded = self.embedding(input_tensor)

        lstm_output, (hidden_state, cell_state) = (
            self.lstm(embedded)
        )

        return hidden_state, cell_state

In [9]:
class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_size,
        hidden_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )

        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size
        )

        self.output_layer = nn.Linear(
            hidden_size,
            vocab_size
        )


    def forward(
        self,
        decoder_input,
        hidden_state,
        cell_state
    ):

        embedded = self.embedding(decoder_input)

        lstm_output, (hidden_state, cell_state) = (
            self.lstm(
                embedded,
                (hidden_state, cell_state)
            )
        )

        logits = self.output_layer(lstm_output)

        return logits, hidden_state, cell_state

In [10]:
embedding_size = 8
hidden_size = 64


encoder = Encoder(
    vocab_size=len(input_characters),
    embedding_size=embedding_size,
    hidden_size=hidden_size
).to(device)


decoder = Decoder(
    vocab_size=len(output_tokens),
    embedding_size=embedding_size,
    hidden_size=hidden_size
).to(device)


print(encoder)
print()
print(decoder)

Encoder(
  (embedding): Embedding(37, 8)
  (lstm): LSTM(8, 64)
)

Decoder(
  (embedding): Embedding(13, 8)
  (lstm): LSTM(8, 64)
  (output_layer): Linear(in_features=64, out_features=13, bias=True)
)


In [11]:
loss_function = nn.CrossEntropyLoss()


parameters = (
    list(encoder.parameters())
    +
    list(decoder.parameters())
)


optimizer = optim.Adam(
    parameters,
    lr=0.003
)

In [12]:
# Understand one complete forward pass

input_text = "29 december 2029"
target_text = "2029-12-29"


# -------------------
# Encoder
# -------------------

input_tensor = encode_input(input_text)

hidden_state, cell_state = encoder(
    input_tensor
)


# -------------------
# Decoder preparation
# -------------------

decoder_input_tensor, decoder_target_tensor = (
    prepare_decoder_sequences(target_text)
)


# -------------------
# Decoder
# -------------------

logits, _, _ = decoder(
    decoder_input_tensor,
    hidden_state,
    cell_state
)


# -------------------
# Loss
# -------------------

loss = loss_function(
    logits,
    decoder_target_tensor
)


print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Logits shape: torch.Size([11, 13])
Loss: 2.6320366859436035


In [13]:
number_of_epochs = 10


for epoch in range(number_of_epochs):

    encoder.train()
    decoder.train()

    random.shuffle(train_data)

    total_loss = 0


    for input_text, target_text in train_data:

        # --------------------------------
        # 1. Clear old gradients
        # --------------------------------

        optimizer.zero_grad()


        # --------------------------------
        # 2. Prepare encoder input
        # --------------------------------

        input_tensor = encode_input(
            input_text
        )


        # --------------------------------
        # 3. Encoder forward pass
        # --------------------------------

        hidden_state, cell_state = encoder(
            input_tensor
        )


        # --------------------------------
        # 4. Prepare decoder data
        # --------------------------------

        (
            decoder_input_tensor,
            decoder_target_tensor
        ) = prepare_decoder_sequences(
            target_text
        )


        # --------------------------------
        # 5. Decoder forward pass
        # --------------------------------

        logits, _, _ = decoder(
            decoder_input_tensor,
            hidden_state,
            cell_state
        )


        # --------------------------------
        # 6. Calculate loss
        # --------------------------------

        loss = loss_function(
            logits,
            decoder_target_tensor
        )


        # --------------------------------
        # 7. Backpropagation
        # --------------------------------

        loss.backward()


        # Optional protection against
        # very large LSTM gradients
        torch.nn.utils.clip_grad_norm_(
            parameters,
            max_norm=1.0
        )


        # --------------------------------
        # 8. Update parameters
        # --------------------------------

        optimizer.step()


        total_loss += loss.item()


    average_loss = total_loss / len(train_data)


    print(
        f"Epoch {epoch + 1:2d} "
        f"| Training Loss: {average_loss:.4f}"
    )

Epoch  1 | Training Loss: 0.5488
Epoch  2 | Training Loss: 0.1874
Epoch  3 | Training Loss: 0.0657
Epoch  4 | Training Loss: 0.0229
Epoch  5 | Training Loss: 0.0116
Epoch  6 | Training Loss: 0.0103
Epoch  7 | Training Loss: 0.0003
Epoch  8 | Training Loss: 0.0295
Epoch  9 | Training Loss: 0.0096
Epoch 10 | Training Loss: 0.0038


In [14]:
def predict_date(input_text, max_length=20):

    encoder.eval()
    decoder.eval()

    with torch.no_grad():

        # ----------------------
        # Encoder
        # ----------------------

        input_tensor = encode_input(
            input_text
        )

        hidden_state, cell_state = encoder(
            input_tensor
        )


        # ----------------------
        # Decoder starts here
        # ----------------------

        current_token_id = (
            output_token_to_id["<START>"]
        )

        current_token = torch.tensor(
            [current_token_id],
            dtype=torch.long,
            device=device
        )

        predicted_characters = []


        for _ in range(max_length):

            logits, hidden_state, cell_state = decoder(
                current_token,
                hidden_state,
                cell_state
            )


            # Decoder processed one token,
            # so take the last output
            token_scores = logits[-1]


            predicted_id = torch.argmax(
                token_scores
            ).item()


            predicted_token = (
                output_id_to_token[predicted_id]
            )


            # Stop generation
            if predicted_token == "<END>":
                break


            predicted_characters.append(
                predicted_token
            )


            # Feed prediction back to decoder
            current_token = torch.tensor(
                [predicted_id],
                dtype=torch.long,
                device=device
            )


    return "".join(predicted_characters)

In [15]:
examples = [
    "9 june 2027",
    "29 december 2029",
    "17 january 2025",
    "3 march 2024",
    "21 september 2028"
]


for example in examples:

    prediction = predict_date(example)

    print(
        f"{example:20} --> {prediction}"
    )

9 june 2027          --> 2027-06-09
29 december 2029     --> 2029-12-29
17 january 2025      --> 2025-01-17
3 march 2024         --> 2024-03-03
21 september 2028    --> 2028-09-21


In [16]:
correct = 0

mistakes = []


for input_text, expected_output in test_data:

    prediction = predict_date(input_text)

    if prediction == expected_output:

        correct += 1

    else:

        mistakes.append(
            (
                input_text,
                expected_output,
                prediction
            )
        )


accuracy = correct / len(test_data)


print(
    f"Test accuracy: {accuracy * 100:.2f}%"
)

print(
    f"Correct: {correct}/{len(test_data)}"
)

Test accuracy: 100.00%
Correct: 403/403


In [17]:
print("\nSome incorrect predictions:\n")

for item in mistakes[:10]:

    input_text, expected, predicted = item

    print("Input    :", input_text)
    print("Expected :", expected)
    print("Predicted:", predicted)
    print()


Some incorrect predictions:



In [18]:
torch.save(
    {
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "input_char_to_id": input_char_to_id,
        "output_token_to_id": output_token_to_id,
        "embedding_size": embedding_size,
        "hidden_size": hidden_size,
    },
    "date_seq2seq_model.pt"
)

print("Model saved.")

Model saved.
